# Player-only detector (YOLO26s, fine-tuned on SportsMOT volleyball)

SportsMOT labels only the players on court, so fine-tuning on its volleyball training sequences teaches the
detector to leave out referees, line judges, coaches, the bench and spectators (change
`filter-on-court-players`, docs/results/player-tracking.md). The data are downloaded and filtered here (15 of the
45 training sequences, ~6.7 GB streamed from Hugging Face; CC BY-NC 4.0, research use, not redistributed) and
converted by `scripts/build_sportsmot_detection.py` from the project repository. SportsMOT's volleyball
validation sequences are not used: they stay for the tracking evaluation.

Runs on **Kaggle** (GPU + Internet on; `kaggle kernels push`) or **Colab** (T4; checkpoints on Drive).
The weights land in the outputs as `player_yolo26s_sportsmot.pt`.


In [ ]:
import os, glob, shutil, subprocess, sys
KAGGLE = os.path.exists('/kaggle')
WORK = '/tmp/work'  # data and repo stay out of the outputs
if KAGGLE:
    OUT = '/kaggle/working'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/volleyball/models'
RUNS = f'{OUT}/runs'
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!pip -q install ultralytics
!git clone -q --depth 1 https://github.com/DL-Volleyball-Analysis/volleyball-analysis {WORK}/repo
import importlib.util
spec = importlib.util.spec_from_file_location('b', f'{WORK}/repo/scripts/build_sportsmot_detection.py')
b = importlib.util.module_from_spec(spec); spec.loader.exec_module(b)
print(len(b.VOLLEYBALL_TRAIN), 'volleyball training sequences')


In [ ]:
# stream the training archive and keep only the volleyball sequences (no 6.7 GB file on disk)
URL = 'https://huggingface.co/datasets/MCG-NJU/SportsMOT/resolve/main/dataset/train.tar'
SRC = f'{WORK}/sportsmot'
os.makedirs(SRC, exist_ok=True)
patterns = ' '.join(f"'*{s}/*'" for s in b.VOLLEYBALL_TRAIN)
!curl -sL --retry 5 "{URL}" | tar -x -C {SRC} --wildcards {patterns}
dirs = {os.path.basename(p): p for p in glob.glob(f'{SRC}/**/v_*', recursive=True) if os.path.isdir(p) and os.path.basename(p) in b.VOLLEYBALL_TRAIN}
print(len(dirs), 'sequences extracted')
assert len(dirs) == 15, sorted(set(b.VOLLEYBALL_TRAIN) - set(dirs))


In [ ]:
from pathlib import Path
src = Path(next(iter(dirs.values()))).parent
DET = f'{WORK}/det'
print('images:', b.build(src, Path(DET), b.VOLLEYBALL_TRAIN, every=3))
cfg = f'{DET}/data.yaml'


In [ ]:
from ultralytics import YOLO
RUN_NAME = 'player_sportsmot_v1'
last = f'{RUNS}/{RUN_NAME}/weights/last.pt'
import torch
finished = os.path.exists(last) and torch.load(last, map_location='cpu', weights_only=False).get('epoch') == -1
if finished:
    print('training already finished:', last)
elif os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    # same detector and input size as the installed one (YOLO26s, 960 px), one class: player
    YOLO('yolo26s.pt').train(data=cfg, epochs=60, imgsz=960, batch=16, patience=15, single_cls=True,
                             project=RUNS, name=RUN_NAME, exist_ok=True)


In [ ]:
import json
best = f'{RUNS}/{RUN_NAME}/weights/best.pt'
m = YOLO(best).val(data=cfg, imgsz=960, batch=16)
res = {'valid_map50': float(m.box.map50), 'valid_map50_95': float(m.box.map), 'precision': float(m.box.mp), 'recall': float(m.box.mr)}
print(json.dumps(res, indent=1))
shutil.copy(best, f'{OUT}/player_yolo26s_sportsmot.pt')
json.dump(res, open(f'{OUT}/player_sportsmot_v1_valid.json', 'w'), indent=1)
print('saved to', OUT)
